# Lab Exercise: One Dataset, Five Techniques
This notebook demonstrates how the **question you ask**, not the dataset itself, dictates which machine learning technique is applied.

We will run five entirely different machine learning techniques (Regression, Classification, Clustering, Anomaly Detection, and Dimensionality Reduction) on the **exact same 14-customer dataset**.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler

# The Dataset: 14 customers with tenure, spend, and support tickets
df = pd.DataFrame({
    "tenure_months":  [2, 3, 4, 5, 8, 10, 12, 15, 18, 20, 24, 30,  6, 36],
    "monthly_spend":  [20, 25, 22, 30, 40, 45, 50, 55, 62, 65, 70, 80, 95, 140],
    "support_tickets": [5, 4, 6, 3, 2, 1, 2, 1, 0, 1, 0, 1,  8,  0],
    "churned":        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,  1,  0],
})
X = df[["tenure_months", "monthly_spend", "support_tickets"]]

print("THE DATA — 14 customers, 3 features, 1 label")
print(df.to_string(index=False))

### 1. Regression (Predicting Continuous Values)
**Question:** *"How much will this customer spend?"* $\to$ predicting a continuous number on a scale.

In [ ]:
reg = LinearRegression().fit(df[["tenure_months"]], df["monthly_spend"])
print(f"Fitted Line: spend = {reg.coef_[0]:.2f} x tenure + {reg.intercept_:.2f}")
print(f"R^2 on training data: {reg.score(df[['tenure_months']], df['monthly_spend']):.3f}")

at_14 = pd.DataFrame({"tenure_months": [14]})
print(f"Predicted monthly spend at 14 months: ${reg.predict(at_14)[0]:.2f}")

### 2. Classification (Predicting Categories)
**Question:** *"Will this customer churn?"* $\to$ predicting a discrete yes/no label from a fixed set.

In [ ]:
clf = LogisticRegression(max_iter=1000).fit(X, df["churned"])
print(f"Training Accuracy: {clf.score(X, df['churned']):.3f}")

new_customer = pd.DataFrame([[3, 24, 5]], columns=X.columns)
prediction = clf.predict(new_customer)[0]
probability = clf.predict_proba(new_customer)[0][1]
print(f"New Customer prediction (3mo, $24, 5 tickets) -> {'CHURN' if prediction else 'STAY'} (probability of churn = {probability:.2f})")

### 3. Clustering (Grouping Unlabeled Data)
**Question:** *"What natural groups of customers exist?"* $\to$ unsupervised grouping. Notice that we never show the `churned` labels to the model!

In [ ]:
Xs = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=3, random_state=0, n_init=10).fit(Xs)
df["segment"] = km.labels_

print("--- Cluster Segment Profiles ---")
print(df.groupby("segment")[["tenure_months", "monthly_spend", "support_tickets"]].mean().round(1).to_string())
print("\nNote: Segment 0 averages 4.0 months tenure and 5.2 tickets—successfully isolating the churners without ever seeing the labels!")

### 4. Anomaly Detection (Finding the Rare/Outlier cases)
**Question:** *"Which customers do not belong?"* $\to$ identifying normal vs. outlier entries.

In [ ]:
iso = IsolationForest(contamination=0.15, random_state=0).fit(X)
flags = iso.predict(X)
outliers = df.loc[flags == -1, ["tenure_months", "monthly_spend", "support_tickets"]]

print(f"Flagged {len(outliers)} of {len(df)} customers as anomalous outliers:")
print(outliers.to_string(index=False))

### 5. Dimensionality Reduction (Fewer Columns)
**Question:** *"Can we compress 3 columns to 2?"* $\to$ retaining maximum variance in fewer dimensions.

In [ ]:
pca = PCA(n_components=2).fit(Xs)
ratios = pca.explained_variance_ratio_
print(f"Variance kept by Principal Component 1: {ratios[0]:.1%}")
print(f"Variance kept by Principal Component 2: {ratios[1]:.1%}")
print(f"Total retained information with 2 components: {ratios.sum():.1%}")

### 6. Lab Experiments (Things to Try)

#### Experiment 1: Number of Clusters
1. Change `n_clusters` in KMeans to 2, and then 4.
2. Observe how the averages redistribute. Notice why clustering is hard to evaluate objectively—there is no single "right" number of unlabeled groups.

#### Experiment 2: Remove Outliers for Regression
1. Observe how the anomalous rows (index 12: 6 months/$95, index 13: 36 months/$140) pull the OLS line off.
2. Drop these two rows from `df` and re-fit the LinearRegression.
3. Observe how your $R^2$ score jumps significantly once the data noise is removed!